# Notebook 04 — AI Summarisation
Runs the full pipeline: geocode → fetch storms → evaluate threat → compose context → Mistral summary.

**Requirements:**
- `MISTRAL_API_KEY` set in `../.env`
- `BRAVE_API_KEY` set in `../.env` (for news; optional)

**Docs:**
- Mistral API: https://docs.mistral.ai/api/
- Mistral models: https://docs.mistral.ai/getting-started/models/

In [ ]:
import sys, json
sys.path.insert(0, '..')

from geocoder import geocode_user_location
from data_fetcher import (
    get_active_storms, fetch_storm_feeds,
    parse_hurricane_gis, fetch_noaa_storm_surge,
    query_hurricane_news,
)
from gis_processor import is_within_threat_zone, ThreatResult
from ai_summarizer import (
    compose_mistral_context,
    generate_hurricane_summary,
    generate_threat_explanation,
)

print('Imports OK')

## 1. Set Location & Geocode

In [ ]:
USER_LOCATION = 'Miami, FL'   # <-- change this

geo = geocode_user_location(USER_LOCATION)
if not geo['success']:
    raise ValueError(f'Could not geocode "{USER_LOCATION}"')

lat, lon = geo['lat'], geo['lon']
print(f'Location: {geo["display_name"]}')
print(f'Lat/Lon:  {lat:.4f}, {lon:.4f}')

## 2. Fetch All Active Storms

In [ ]:
storms = get_active_storms()
print(f'{len(storms)} active storm(s):')
for s in storms:
    print(f"  {s['storm_type']} {s['name']} ({s['basin']})")

## 3. Gather Data for Each Storm

In [ ]:
THREAT_ORDER  = {'None': 0, 'Low': 1, 'Moderate': 2, 'High': 3, 'Extreme': 4}
all_rss_texts = {}
all_news      = []
all_threats   = []
worst_threat  = ThreatResult()

for storm in storms:
    feeds  = fetch_storm_feeds(storm)
    gis    = parse_hurricane_gis(storm)
    surge  = fetch_noaa_storm_surge(storm)
    threat = is_within_threat_zone(lat, lon, gis, surge.get('surge_gdf'))
    news   = query_hurricane_news(f"{storm['name']} hurricane {USER_LOCATION}", count=3)

    all_rss_texts[storm['storm_id']] = feeds['advisory_text']
    all_news.extend(news)
    all_threats.append((storm, threat))

    if THREAT_ORDER.get(threat.threat_level, 0) > THREAT_ORDER.get(worst_threat.threat_level, 0):
        worst_threat = threat

    print(f"{storm['name']}: threat={threat.threat_level}, news={len(news)} articles")

print(f'\nWorst threat overall: {worst_threat.threat_level}')

## 4. Compose Mistral Context (inspect before sending)

In [ ]:
ctx = compose_mistral_context(
    storms=[s for s, _ in all_threats],
    rss_texts=all_rss_texts,
    news=all_news,
    threat_result=worst_threat,
    user_location=USER_LOCATION,
)

# Print context — this is what gets sent to Mistral
print(json.dumps(ctx, indent=2))

## 5. Generate AI Hurricane Summary (Mistral Large)
**Cost:** ~0.003 € per call from the 15 € free credit pool.

In [ ]:
if not storms:
    print('No active storms — AI summary not required.')
else:
    print('Calling Mistral Large...')
    try:
        summary = generate_hurricane_summary(ctx)
        print('\n' + '='*60)
        print('AI SAFETY BRIEFING')
        print('='*60)
        print(summary)
    except Exception as e:
        print(f'Error: {e}')
        print('Ensure MISTRAL_API_KEY is set in ../.env')

## 6. Generate Threat Explanation (Mistral Small — cheaper)

In [ ]:
if storms:
    try:
        explanation = generate_threat_explanation(ctx)
        print('THREAT EXPLANATION (plain language):')
        print(explanation)
    except Exception as e:
        print(f'Error: {e}')

## 7. API Cost Estimate for This Run

In [ ]:
ctx_chars  = len(json.dumps(ctx))
ctx_tokens = ctx_chars // 4  # rough estimate: 1 token ≈ 4 chars

print('Estimated API usage for this analysis:')
print(f'  Mistral context tokens (input): ~{ctx_tokens:,}')
print(f'  Mistral output tokens (est.):   ~400')
print(f'  Total tokens:                   ~{ctx_tokens+400:,}')
print(f'  Brave Search queries:           {len(storms)} (one per storm)')
print(f'  Nominatim queries:              1')
print()
print('Free credit impact:')
print(f'  Mistral cost (est.):   ~€{(ctx_tokens+400) * 0.000003:.4f} of €15 budget')
print(f'  Brave Search cost:     ~${len(storms) * 0.005:.3f} of $5 budget')